# Crypto-Forex Trading: Position Sizing & Risk Calculator


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Crypto-Forex Trading: Position Sizing & Risk Calculator

This notebook demonstrates professional position-sizing and risk-management techniques for crypto-forex trading, inspired by best practices in crypto forex trading guides.

**Key concepts covered:**
- Fixed fractional position sizing
- Risk-reward ratio calculation
- Leverage impact analysis
- Liquidation price computation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define trading parameters
account_balance = 10000  # USD
risk_per_trade = 0.02   # 2% risk per trade
leverage = 5            # 5x leverage

# Example crypto-forex pair: BTC/USD
entry_price = 42500
exit_price_profit = 43500
exit_price_loss = 41500

print("=" * 60)
print("CRYPTO-FOREX TRADING POSITION SIZING CALCULATOR")
print("=" * 60)
print(f"Account Balance: ${account_balance:,.2f}")
print(f"Risk Per Trade: {risk_per_trade*100:.1f}%")
print(f"Leverage: {leverage}x")
print(f"\nTrade Setup: BTC/USD")
print(f"Entry Price: ${entry_price:,.2f}")
print(f"Profit Target: ${exit_price_profit:,.2f}")
print(f"Stop Loss: ${exit_price_loss:,.2f}")

In [ ]:
# Calculate dollar risk (max loss acceptable)
dollar_risk = account_balance * risk_per_trade
print(f"\n--- POSITION SIZING ---")
print(f"Dollar Risk (max loss): ${dollar_risk:,.2f}")

# Calculate pips/points risk
price_risk = entry_price - exit_price_loss  # Distance to stop loss
print(f"Price Risk (entry to stop): ${price_risk:,.2f}")

# Calculate contract/lot size for fixed fractional method
# Position size = (Dollar Risk) / (Price Risk per unit)
position_size_contracts = dollar_risk / price_risk
print(f"\nPosition Size (contracts): {position_size_contracts:.4f}")

# Notional value with leverage
notional_value = position_size_contracts * entry_price * leverage
margin_required = notional_value / leverage  # Actual margin collateral needed
print(f"Notional Exposure (with {leverage}x leverage): ${notional_value:,.2f}")
print(f"Margin Required: ${margin_required:,.2f}")
print(f"Margin % of Account: {(margin_required/account_balance)*100:.2f}%")

In [ ]:
# Calculate profit and loss scenarios
profit_per_unit = exit_price_profit - entry_price
loss_per_unit = entry_price - exit_price_loss

total_profit = position_size_contracts * profit_per_unit
total_loss = position_size_contracts * loss_per_unit

print(f"\n--- PROFIT/LOSS SCENARIO ---")
print(f"If price reaches ${exit_price_profit:,.2f} (profit target):")
print(f"  Profit per unit: ${profit_per_unit:,.2f}")
print(f"  Total Profit: ${total_profit:,.2f}")
print(f"  ROI: {(total_profit/account_balance)*100:.2f}%")

print(f"\nIf price reaches ${exit_price_loss:,.2f} (stop loss):")
print(f"  Loss per unit: ${loss_per_unit:,.2f}")
print(f"  Total Loss: ${total_loss:,.2f}")
print(f"  ROI: {(total_loss/account_balance)*100:.2f}%")

# Risk-Reward Ratio
risk_reward_ratio = (exit_price_profit - entry_price) / (entry_price - exit_price_loss)
print(f"\nRisk-Reward Ratio: 1:{risk_reward_ratio:.2f}")

In [ ]:
# Liquidation price analysis
# For leveraged positions: liquidation occurs when collateral < maintenance margin
# Simplified: Liquidation Price = Entry Price - (Margin / Position Size)

aintenance_margin_ratio = 0.05  # 5% maintenance margin (typical for crypto-forex)
liquidation_price = entry_price - ((margin_required * maintenance_margin_ratio) / position_size_contracts)

print(f"\n--- LIQUIDATION RISK ANALYSIS ---")
print(f"Current Entry Price: ${entry_price:,.2f}")
print(f"Liquidation Price (5% maintenance): ${liquidation_price:,.2f}")
print(f"Distance to Liquidation: ${entry_price - liquidation_price:,.2f}")
print(f"Liquidation Risk (% from entry): {((entry_price - liquidation_price)/entry_price)*100:.2f}%")

if liquidation_price < exit_price_loss:
    print(f"✓ Stop loss (${exit_price_loss:,.2f}) is above liquidation price (safe)")
else:
    print(f"✗ WARNING: Stop loss is below liquidation price (risky!)")

In [ ]:
# Multi-scenario comparison table
leverage_levels = [1, 2, 5, 10, 20]
scenarios = []

for lev in leverage_levels:
    notional = position_size_contracts * entry_price * lev
    margin = notional / lev
    margin_pct = (margin / account_balance) * 100
    liq_price = entry_price - ((margin * 0.05) / position_size_contracts)
    
    scenarios.append({
        'Leverage': f"{lev}x",
        'Notional Value': f"${notional:,.0f}",
        'Margin Required': f"${margin:,.0f}",
        'Margin % of Account': f"{margin_pct:.1f}%",
        'Liquidation Price': f"${liq_price:,.0f}",
        'Risk Level': 'Low' if margin_pct < 10 else 'Medium' if margin_pct < 20 else 'High'
    })

df_scenarios = pd.DataFrame(scenarios)
print(f"\n--- LEVERAGE COMPARISON TABLE ---")
print(df_scenarios.to_string(index=False))

In [ ]:
# Visualization: Position sizing across account sizes
account_sizes = np.array([1000, 5000, 10000, 25000, 50000])
position_sizes_5x = []
margin_percentages = []

for acc in account_sizes:
    dr = acc * risk_per_trade
    pos_size = dr / price_risk
    not_val = pos_size * entry_price * 5
    margin = not_val / 5
    position_sizes_5x.append(margin)
    margin_percentages.append((margin / acc) * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Chart 1: Margin required vs account size
ax1.bar(account_sizes, position_sizes_5x, color='#3b82f6', alpha=0.7)
ax1.set_xlabel('Account Size (USD)')
ax1.set_ylabel('Margin Required (USD)')
ax1.set_title('Margin Required vs Account Size (5x Leverage)')
ax1.grid(axis='y', alpha=0.3)

# Chart 2: Margin as % of account
ax2.plot(account_sizes, margin_percentages, marker='o', linewidth=2, markersize=8, color='#3b82f6')
ax2.axhline(y=20, color='red', linestyle='--', label='Caution Threshold (20%)')
ax2.set_xlabel('Account Size (USD)')
ax2.set_ylabel('Margin as % of Account')
ax2.set_title('Margin Usage Ratio')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nChart: Position sizing scales consistently across account sizes at 2% risk per trade.")

## Key Takeaways

1. **Fixed Fractional Sizing**: Position size = (Account × Risk %) / Price Risk. This automatically scales with account growth.

2. **Leverage Amplifies Both Gains and Risk**: 5x leverage multiplies returns but also increases liquidation risk. Monitor margin usage.

3. **Risk-Reward Matters**: A 1:2 or better risk-reward ratio improves long-term profitability despite win-rate.

4. **Liquidation Price**: Always check that your stop-loss is well above liquidation price. Use conservative maintenance margin assumptions.

5. **Professional Practice**: Risking 1-2% per trade on crypto-forex is standard for portfolio preservation.

For more on crypto-forex trading mechanics, see the complete professional guide.

---

## Reference

[protraderdaily](https://protraderdaily.com/crypto/what-is-crypto-forex-trading)
